## Data Cleaning Project: Cafe Sales Dataset

## Load the Dataset 

In [103]:
import pandas as pd
import numpy as np

In [104]:
# Load dataset
df = pd.read_csv("dirty_cafe_sales.csv")

In [105]:
# Display first 5 rows
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [106]:
# There are ERROR and UNKNOWN which are to be converted into NaN 

In [107]:
df.shape

(10000, 8)

In [108]:
# Basic Information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [109]:
# Null Values
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [110]:
# Duplicate Rows
df.duplicated().sum()


np.int64(0)

In [111]:
# Data Types
df.dtypes

Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

In [112]:
# Check Value Range Anomalies
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in numeric_cols:
    temp = pd.to_numeric(df[col], errors="coerce")

    print(col)
    print("Minimum:", temp.min())
    print("Maximum:", temp.max())
    print("")

# erroes="coerce" turns Errors into NaN, if any

Quantity
Minimum: 1.0
Maximum: 5.0

Price Per Unit
Minimum: 1.0
Maximum: 5.0

Total Spent
Minimum: 1.0
Maximum: 25.0



## Missing Data Handling

In [113]:
# Item 
# Catagorical value - Mode Imputation
# Replace invalid values with NaN
df["Item"] = df["Item"].replace(["ERROR", "UNKNOWN"], np.nan)
df["Item"] = df["Item"].fillna(df["Item"].mode()[0])

In [114]:
#Quantity
#It has 138 missing values and is numerical 

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
# Converting Data to Numeric and ERROR, INVALID to NaN


In [115]:
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())

#Invalid values were converted to missing values using `pd.to_numeric(errors="coerce")`. Missing values were then replaced with the median because the median is less affected by extreme values than the mean.

In [116]:
# Price Per Unit : Numeric - Median Imputation
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"],errors="coerce")
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Price Per Unit"].median())

In [117]:
# Total Spent : Numeric - Median Imputation
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")
df["Total Spent"] = df["Total Spent"].fillna(df["Total Spent"].median())

In [118]:
# Payment Method : Catagorical-Mode Imputation
df["Payment Method"] = df["Payment Method"].replace(["ERROR", "UNKNOWN"],np.nan)
df["Payment Method"] = df["Payment Method"].fillna(df["Payment Method"].mode()[0])

In [119]:
# Location : Catagorical-Mode Imputation
df["Location"] = df["Location"].replace(["ERROR", "UNKNOWN"],np.nan)
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

In [120]:
# Transaction Date : Catagorical-Mode Imputation
# Convert invalid dates to NaT
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

# Remove "rows" with missing dates
df = df.dropna(subset=["Transaction Date"])

The Transaction Date column contains date values.

Strategy Used: Row Deletion

Invalid dates were converted to missing values. Rows containing missing dates were removed because assigning incorrect dates could affect the accuracy of the dataset.

## Duplicates Removal

In [121]:
df.duplicated().sum()

np.int64(0)

Duplicate rows were identified using the `duplicated()` function.
Duplicate records can lead to inaccurate analysis and should be removed. In this dataset, no duplicate rows were found, so no rows were removed.

In [122]:
# Count duplicate rows
duplicates = df.duplicated().sum()

print("Number of duplicate rows:", duplicates)

# Remove duplicate rows
df = df.drop_duplicates()

print("Number of rows after removing duplicates:", len(df))

Number of duplicate rows: 0
Number of rows after removing duplicates: 9540


## Standardisation

Data was standardised to ensure consistent formatting throughout the dataset.

**Actions Performed:**
- Removed leading and trailing spaces from categorical columns.
- Converted text values to title case (e.g., `coffee` → `Coffee`).
- Converted the **Transaction Date** column to the datetime format.

In [123]:
# Remove leading/trailing spaces and standardize text format

categorical_cols = ["Item", "Payment Method", "Location"]

for col in categorical_cols:
    df[col] = df[col].str.strip().str.title()

# Convert Transaction Date to datetime format
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"]) 

# .Strip() removes leading/trailing spaces and .title() changes it to title case

## Outlier Detection

Outliers were identified in the numeric columns using the **Interquartile Range (IQR)** method.

**Columns Checked:**
- Quantity
- Price Per Unit
- Total Spent

**Decision:** Outliers were **retained** because they may represent genuine customer purchases rather than data entry errors. Removing them could result in the loss of important business information.

In [124]:
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"{col}")
    print("Number of outliers:", len(outliers))
    print("")

Quantity
Number of outliers: 0

Price Per Unit
Number of outliers: 0

Total Spent
Number of outliers: 250



## Data Type Correction

The data types of all columns were checked and corrected to ensure consistency.

**Changes Made:**
- Transaction ID → String
- Item → String
- Quantity → Integer
- Price Per Unit → Float
- Total Spent → Float
- Payment Method → String
- Location → String
- Transaction Date → Datetime

In [125]:
# Convert columns to appropriate data types

df["Transaction ID"] = df["Transaction ID"].astype(str)
df["Item"] = df["Item"].astype(str)
df["Quantity"] = df["Quantity"].astype(int)
df["Price Per Unit"] = df["Price Per Unit"].astype(float)
df["Total Spent"] = df["Total Spent"].astype(float)
df["Payment Method"] = df["Payment Method"].astype(str)
df["Location"] = df["Location"].astype(str)
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])

In [126]:
#Check
df.dtypes

Transaction ID                 str
Item                           str
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

## Before vs After Cleaning Summary

The table below compares the dataset before and after the data cleaning process.

In [127]:
summary = pd.DataFrame({
    "Metric": [
        "Total Null Values",
        "Duplicate Rows",
        "Row Count",
        "Correct Data Types"
    ],
    "Before Cleaning": [
        6826,
        0,
        10000,
        "No"
    ],
    "After Cleaning": [
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        len(df),
        "Yes"
    ]
})

summary

,Metric,Before Cleaning,After Cleaning
0,Total Null Values,6826,0
1,Duplicate Rows,0,0
2,Row Count,10000,9540
3,Correct Data Types,No,Yes


## Save Cleaned Dataset

The cleaned dataset was saved as a new CSV file named **cleaned_cafe_sales.csv** for future analysis and use.

In [128]:
df.to_csv("cleaned_cafe_sales.csv", index=False)

By default, pandas saves the row index as an extra column in the CSV. Using:

index=False

prevents that extra index column from being written, so the saved file contains only your dataset columns.